### Objective and data sources

Check whether the monitoring data from PVDAQ System 2107 can be used to investigate periods of low solar generation.

- **AC meter data** — average AC power (kW) at 15-minute intervals.
- **Irradiance data** — sunlight reaching the plane of the panels (W/m²).
- **System metadata** — site details, sensor definitions and time zone.

Both data files use `measured_on` for timestamps.

### Load and inspect the monitoring data

In [63]:
import pandas as pd

data_path = "./data/raw/pvdaq_2107/"
irradiance_data = pd.read_csv(data_path + "2107_irradiance_data_2024.csv")
meter_data = pd.read_csv(data_path + "2107_meter_15m_data_2024.csv")
irradiance_data.tail()

,measured_on,poa_irradiance_o_149574
73072,2024-11-01 23:10:00,0.0
73073,2024-11-01 23:15:00,0.0
73074,2024-11-01 23:20:00,0.0
73075,2024-11-01 23:35:00,0.0
73076,2024-11-01 23:55:00,0.0


### Audit timestamps and measurement quality

The AC meter records at 15-minute intervals. Irradiance readings are most commonly 5 minutes apart, with frequent gaps of 10, 15 and 20 minutes. The longest gap between irradiance readings was 43 hours and 35 minutes, from 17 October at 16:35 to 19 October at 12:10.

On 10 March, the spring daylight-saving transition made a 15-minute elapsed interval appear as 75 minutes in both streams. Once the timestamps were localised to Pacific time, the meter interval matched its normal cadence. The irradiance interval was still longer than its most common 5-minute spacing.

On 3 November, the meter file contained one set of four readings during the repeated 01:00–01:59 hour. Their UTC offsets could not be determined from the CSV. The irradiance file ends on 1 November, so these four readings fall outside the shared analysis window.

In [64]:
meter_data["measured_on"] = pd.to_datetime(meter_data["measured_on"], errors="coerce")

# Count missing timestamps and values that could not be parsed
print("Missing or invalid timestamps:")
print(meter_data.isna().sum())

timestamps = meter_data["measured_on"].dropna().sort_values()

print("\nMeasured and count the gap in timestamps:")
print(timestamps.diff().value_counts())

print("\nWhere the gap happend:")
print(timestamps[timestamps.diff() > pd.Timedelta(minutes=15)])

Missing or invalid timestamps:
measured_on                                   0
meter_revenue_grade_ac_output_meter_149578    0
dtype: int64

Measured and count the gap in timestamps:
measured_on
0 days 00:15:00    32153
0 days 01:15:00        1
Name: count, dtype: int64

Where the gap happend:
6631   2024-03-10 03:00:00
Name: measured_on, dtype: datetime64[us]


In [65]:
irradiance_data["measured_on"] = pd.to_datetime(irradiance_data["measured_on"], errors="coerce")

# Count missing timestamps and values that could not be parsed
print("Missing or invalid timestamps:")
print(irradiance_data.isna().sum())

timestamps = irradiance_data["measured_on"].dropna().sort_values()

print("\nMeasured and count the gap in timestamps:")
print(timestamps.diff().value_counts())

print("\nWhere the gap happend:")
print(timestamps[timestamps.diff() > pd.Timedelta(minutes=20)])

Missing or invalid timestamps:
measured_on                0
poa_irradiance_o_149574    0
dtype: int64

Measured and count the gap in timestamps:
measured_on
0 days 00:05:00    64976
0 days 00:10:00     3428
0 days 00:15:00     3004
0 days 00:20:00     1663
0 days 01:15:00        1
0 days 04:25:00        1
0 days 02:00:00        1
0 days 01:10:00        1
1 days 19:35:00        1
Name: count, dtype: int64

Where the gap happend:
15878   2024-03-10 03:05:00
25053   2024-04-17 13:00:00
47469   2024-07-16 15:50:00
52354   2024-08-05 11:40:00
69961   2024-10-19 12:10:00
Name: measured_on, dtype: datetime64[us]


In [66]:
timestamps = irradiance_data["measured_on"].dropna().sort_values()
pacific_timestamps = timestamps.dt.tz_localize("America/Los_Angeles")

gap_check = pd.DataFrame({
    "previous_reading": timestamps.shift(),
    "next_reading": timestamps,
    "clock_gap": timestamps.diff(),
    "elapsed_gap": pacific_timestamps.diff(),
})

gap_check.loc[gap_check["clock_gap"] > pd.Timedelta(minutes=20)]

,previous_reading,next_reading,clock_gap,elapsed_gap
15878,2024-03-10 01:50:00,2024-03-10 03:05:00,0 days 01:15:00,0 days 00:15:00
25053,2024-04-17 08:35:00,2024-04-17 13:00:00,0 days 04:25:00,0 days 04:25:00
47469,2024-07-16 13:50:00,2024-07-16 15:50:00,0 days 02:00:00,0 days 02:00:00
52354,2024-08-05 10:30:00,2024-08-05 11:40:00,0 days 01:10:00,0 days 01:10:00
69961,2024-10-17 16:35:00,2024-10-19 12:10:00,1 days 19:35:00,1 days 19:35:00


In [67]:
meter_times = meter_data["measured_on"]

meter_data.loc[
    (meter_times >= "2024-11-03 00:30")
    & (meter_times < "2024-11-03 02:30")
]

,measured_on,meter_revenue_grade_ac_output_meter_149578
29469,2024-11-03 00:30:00,0.0
29470,2024-11-03 00:45:00,0.0
29471,2024-11-03 01:00:00,0.0
29472,2024-11-03 01:15:00,0.0
29473,2024-11-03 01:30:00,0.0
29474,2024-11-03 01:45:00,0.0
29475,2024-11-03 02:00:00,0.0
29476,2024-11-03 02:15:00,0.0


In [69]:
timestamps = meter_data["measured_on"].dropna().sort_values()
meter_pacific = timestamps.dt.tz_localize(
    "America/Los_Angeles",
    ambiguous="NaT",
)

unresolved = meter_pacific.isna()
print("Ambiguous timestamps:", unresolved.sum())  # 4
print(meter_pacific.diff().value_counts())

Ambiguous timestamps: 4
measured_on
0 days 00:15:00    32149
Name: count, dtype: int64


### Plot AC power and irradiance for 1–7 January 2024

### Review one low-output interval
